## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

In [33]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
import time  # 👈 Import nécessaire pour la pause (retry)
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from google.genai import types
from google.genai.errors import APIError  # Pour attraper proprement les erreurs de surcharge

from IPython.display import Markdown, display

In [34]:
# Always remember to do this!
load_dotenv(override=True)

True

In [36]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key not set
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)
Grok API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [37]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [38]:
messages

[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [40]:
# GOOGLE GEMINI API

# 1. Initialisation du client
google_ai_client = genai.Client(api_key=google_api_key)

# 2. Séparation de l'instruction système et nettoyage des rôles
system_instruction = None
formatted_messages = []

for msg in messages:
    if msg["role"] == "system":
        system_instruction = msg["content"]
    else:
        google_role = "user" if msg["role"] == "user" else "model"
        formatted_messages.append(
            types.Content(
                role=google_role,
                parts=[types.Part.from_text(text=msg["content"])]
            )
        )

# 3. Configuration système
config = types.GenerateContentConfig(
    system_instruction=system_instruction
) if system_instruction else None

# 4. Liste des modèles à essayer par ordre de préférence
models_to_try = [
    "gemini-3.8-flash", 
    "gemini-3.5-flash-lite", 
    "gemini-3.7-flash", 
    "gemini-3.6-flash",
    "llama3.2:latest"
]

response = None

for model_name in models_to_try:
    try:
        print(f"Tentative avec le modèle : {model_name}...")
        response = google_ai_client.models.generate_content(
            model=model_name,
            contents=formatted_messages,
            config=config
        )
        print(f"Succès avec {model_name} !")
        break  # On sort de la boucle si l'appel réussit
    except APIError as e:
        # Si c'est une erreur 503 (surcharge), on patiente 2 secondes avant de basculer
        if e.code == 503:
            print(f"Le modèle {model_name} est surchargé (503). Pause de 2s avant le modèle de secours...")
            time.sleep(2)
        else:
            print(f"Erreur API avec {model_name} : {e.message}")
        continue
    except Exception as e:
        print(f"Échec avec {model_name} (Serveur surchargé ou indisponible).")
        continue

# 5. Affichage du résultat si un modèle a répondu
if response:
    question = response.text
    display(Markdown(question))
else:
    print("Tous les modèles testés sont actuellement indisponibles en raison d'une forte demande. Réessayez dans quelques instants.")


Tentative avec le modèle : gemini-3.8-flash...
Le modèle gemini-3.8-flash est surchargé (503). Pause de 2s avant le modèle de secours...
Tentative avec le modèle : gemini-3.5-flash-lite...
Succès avec gemini-3.5-flash-lite !


If a society achieves absolute abundance and eliminates all scarcity, what primary driver replaces economic necessity in giving human life a sense of meaning, and what paradox might arise from this transition? Answer in one sentence.

## Calling LLMs from multiple providers

We are about to call LLMs from many other providers.
They all provide API endpoints that are compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use these endpoints as if we are using OpenAI.

Please note:

I'm going to use lots of LLMs from different providers, but you don't need to! This is only to show their abilities.

In [41]:
# OpenAI Compatible URLs

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROK_BASE_URL = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [42]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!

anthropic = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL) if anthropic_api_key else None
deepseek = OpenAI(api_key=deepseek_api_key, base_url=DEEPSEEK_BASE_URL) if deepseek_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL) if google_api_key else None
groq = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL) if groq_api_key else None
grok = OpenAI(api_key=grok_api_key, base_url=GROK_BASE_URL) if grok_api_key else None
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL) if openrouter_api_key else None
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [43]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [44]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(f"### {model_name}\n{answer}"))

In [45]:
# Fonction pour associer automatiquement un modèle au bon client OpenAPI

def get_client_for_model(model_name):
    model_lower = model_name.lower()

    if "claude" in model_lower or "anthropic" in model_lower:
        return anthropic
    elif "deepseek" in model_lower:
        return deepseek
    elif "gemini" in model_lower:
        return gemini
    elif "groq" in model_lower or "llama-3" in model_lower:
        # Note : Si vous utilisez Groq pour exécuter Llama, ajustez selon vos besoins
        return groq if groq else openrouter 
    elif "grok" in model_lower:
        return grok
    elif "ollama" in model_lower:
        return ollama 
        
    return openrouter if openrouter else ollama

In [46]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

# Boucle d'exécution universelle via l'interface OpenAI
for model_name in models_to_try :
    # Récupération dynamique du client configuré
    client = get_client_for_model(model_name)

    if client is None:
        print(f"Saut de {model_name} : Le client ou la clé API assoiée n'est pas disponible.")
        continue

    try : 
        print(f"Appel de {model_name}...")

        # Configuration des arguments de base universels OpenAI
        kwargs = {
            "model": model_name,
            "messages": messages
        }

        # Le paramètre reasoning_effort provoque des erreurs 400 sur la plupart des API tiers.
        # On ne l'ajoute que pour les modèles de raisonnement compatibles d'OpenAI/DeepSeek si nécessaire.
        if "o1" in model_name or "o3" in model_name:
            kwargs["reasoning_effort"] = "none"

        response = client.chat.completions.create(**kwargs)

        # Extraction standard OpenAI
        answer = response.choices[0].message.content
        record(model_name, answer)
        
       
    except Exception as e:
        print(f"❌ Échec de l'appel pour {model_name} : {e}")

Appel de gemini-3.8-flash...


### gemini-3.8-flash
The intrinsic pursuit of creative self-actualization and personal mastery replaces economic necessity as the primary driver of meaning, generating the paradox that humanity must artificially manufacture struggle and constraint to find purpose in a world that has eradicated them.

Appel de gemini-3.5-flash-lite...


### gemini-3.5-flash-lite
While self-actualization, existential exploration, and the pursuit of intrinsic purpose would replace economic necessity as the primary drivers of meaning, the resulting paradox is that eliminating struggle may strip human achievements of their perceived value, causing an existential crisis where nothing feels truly meaningful precisely because everything is effortless.

Appel de gemini-3.7-flash...


### gemini-3.7-flash
The pursuit of self-actualization, creative mastery, and social connection replaces economic necessity, giving rise to the paradox that eliminating all external struggle and limitation removes the very friction that historically gave human achievement and purpose their perceived value, potentially plunging society into existential malaise.

Appel de gemini-3.6-flash...


### gemini-3.6-flash
Intrinsic self-actualization replaces economic necessity as the primary driver of human meaning, giving rise to the paradox that the total elimination of struggle and constraint may strip achievement of its value, ultimately fueling a widespread crisis of existential purposelessness.

Appel de llama3.2:latest...


### llama3.2:latest
In a society with absolute abundance, freedom from economic necessity is replaced by the primary driver of personal autonomy and creative fulfillment, potentially leading to the paradox of individuals finding meaning and purpose in having too many choices and freedom, rather than feeling the constraints of scarcity that previously drove growth and fulfillment.

In [63]:
model_name = "claude-sonnet-4-6"

response = anthropic.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

AttributeError: 'NoneType' object has no attribute 'chat'

In [64]:
model_name = "gemini-3.1-flash-lite"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

### gemini-3.1-flash-lite
Your observation touches upon the core of **Buridan’s Ass**, the philosophical paradox where an entity, placed perfectly between two identical bundles of hay, starves to death because it lacks a sufficient reason to prefer one over the other.

If we accept your premise—that decision-making requires a directional vector (a value or preference)—we arrive at several profound conclusions regarding the nature of agency and logic:

### 1. The Necessity of the "Arbitrary"
If an entity is purely rational and devoid of pre-existing values, it enters a state of logical paralysis. To "break the symmetry," the entity must introduce an element that is external to the logic of the options themselves. 
* **Stochastic Choice:** The entity might resort to a random number generator (a coin flip). However, randomness is not a "choice" in the teleological sense; it is a surrender of agency to entropy.
* **Meta-Programming:** If the entity is programmed to prioritize "speed of action" over "quality of outcome," it creates a meta-value (efficiency) that breaks the symmetry. But this assumes the pre-existence of a value (efficiency), which contradicts your premise of a value-less entity.

### 2. The Relationship Between "Being" and "Preferring"
Your argument suggests that **agency is not merely the capacity to act, but the capacity to differentiate.** Without the ability to differentiate, an entity is not an agent; it is a mirror. It can only reflect the environment. 
This suggests that "will" is not an addition to consciousness, but the very mechanism by which consciousness distinguishes itself from the environment. To "be" is to define a boundary, and to define a boundary is to prefer the "inside" to the "outside." Therefore, a truly value-less entity would not even have a sense of self, as the self is the ultimate "value" or preference.

### 3. The Infinite Regress of Reasons
If we attempt to provide a reason for every preference, we encounter an infinite regress. Eventually, we hit a bedrock of **axiomatic values**—preferences that are held not because of a deeper reason, but because they are the foundation upon which "reasoning" is built (e.g., the preference for existence over non-existence, or consistency over contradiction). 
If an entity lacks these axioms, it has no ground to stand on. It is essentially an observer with no "lens" through which to focus its attention.

### 4. Determinism vs. Indeterminacy
In a purely deterministic universe, the symmetry is broken by causal history (e.g., the ass is closer to one pile than the other, or one pile is slightly fresher). In this view, "decision-making" is an illusion; we are simply observing the collapse of a state caused by prior physical variables.
In an indeterminate universe, the symmetry might be broken by a quantum fluctuation. Yet, even if a choice is made through quantum randomness, it remains **non-volitional.** 

### The Synthesis: Is "Indifferent Decision" Possible?
If we grant your premise, we can define a **"Decision-Maker"** as: 
> *An entity that imposes a subjective value-gradient onto an objective field of possibilities.*

Without this gradient, you don't have a decision-maker; you have a data-processing void. The "break" in symmetry is the moment the entity creates a preference where none existed. This implies that **will is the act of creating value, not the act of discovering it.** 

**The question for you is:** Do you believe that values are "discovered" properties of the world (e.g., some things are objectively better than others), or are they "created" by the entity solely to resolve this symmetry-breaking problem?

In [ ]:
model_name = "deepseek-v4-flash"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "moonshotai/kimi-k2.6"

response = openrouter.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">Many models on Ollama are FAR too large for your home computer. Be sure to browse the models on the Ollama website. Look to use models that are size 3GB or less unless you know better; llama3.2 is a great first choice. Don't pick models that end in :cloud; that's something different (a cloud inference service, like Groq).
            </span>
        </td>
    </tr>
</table>

In [47]:
!ollama pull llama3.2:latest
# for running the code in the terminal

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [48]:
import requests
requests.get('http://localhost:11434').content

b'Ollama is running'

In [49]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()
for model in models.get("data"):
    print(model.get("id"))

llama3.2:latest
gurubot/tinystories-656k-q8:latest
mapler/gpt2:latest
gemma3:270m
qwen2.5:7b


In [50]:
model_name = "llama3.2:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

### llama3.2:latest
In a society with absolute abundance, social, emotional, and intrinsic needs, such as personal fulfillment, creative expression, and relationships, might become the primary drivers giving human life a sense of meaning, replaced by the paradox that the elimination of economic necessity could inadvertently lead to a society experiencing a collective identity crisis or feeling less purpose-driven as traditional roles of scarcity-driven competition and ambition become redundant.

In [55]:
model_name = "gemma3:270m"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Absolute abundance leads to a nihilistic sense of meaning and purpose, paradoxically replacing economic necessity.


In [57]:
model_name = "qwen2.5:7b"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In a society achieving absolute abundance, the primary driver that might replace economic necessity in giving human life a sense of meaning could be personal purpose and fulfillment, yet this transition might paradoxically create a new scarcity of meaning as traditional drivers and benchmarks for value may diminish.

In [58]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)


10
['gemini-3.8-flash', 'gemini-3.5-flash-lite', 'gemini-3.7-flash', 'gemini-3.6-flash', 'llama3.2:latest', 'llama3.2:latest', 'gemma3:270m', 'gemma3:270m', 'gurubot/tinystories-656k-q8:latest', 'qwen2.5:7b']
['The intrinsic pursuit of creative self-actualization and personal mastery replaces economic necessity as the primary driver of meaning, generating the paradox that humanity must artificially manufacture struggle and constraint to find purpose in a world that has eradicated them.', 'While self-actualization, existential exploration, and the pursuit of intrinsic purpose would replace economic necessity as the primary drivers of meaning, the resulting paradox is that eliminating struggle may strip human achievements of their perceived value, causing an existential crisis where nothing feels truly meaningful precisely because everything is effortless.', 'The pursuit of self-actualization, creative mastery, and social connection replaces economic necessity, giving rise to the paradox

In [60]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n{answer}\n\n")


Competitor: gemini-3.8-flash
The intrinsic pursuit of creative self-actualization and personal mastery replaces economic necessity as the primary driver of meaning, generating the paradox that humanity must artificially manufacture struggle and constraint to find purpose in a world that has eradicated them.


Competitor: gemini-3.5-flash-lite
While self-actualization, existential exploration, and the pursuit of intrinsic purpose would replace economic necessity as the primary drivers of meaning, the resulting paradox is that eliminating struggle may strip human achievements of their perceived value, causing an existential crisis where nothing feels truly meaningful precisely because everything is effortless.


Competitor: gemini-3.7-flash
The pursuit of self-actualization, creative mastery, and social connection replaces economic necessity, giving rise to the paradox that eliminating all external struggle and limitation removes the very friction that historically gave human achievement

In [63]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n"
    together += answer + "\n\n"

In [64]:
print(together)

# Response from competitor 1
The intrinsic pursuit of creative self-actualization and personal mastery replaces economic necessity as the primary driver of meaning, generating the paradox that humanity must artificially manufacture struggle and constraint to find purpose in a world that has eradicated them.

# Response from competitor 2
While self-actualization, existential exploration, and the pursuit of intrinsic purpose would replace economic necessity as the primary drivers of meaning, the resulting paradox is that eliminating struggle may strip human achievements of their perceived value, causing an existential crisis where nothing feels truly meaningful precisely because everything is effortless.

# Response from competitor 3
The pursuit of self-actualization, creative mastery, and social connection replaces economic necessity, giving rise to the paradox that eliminating all external struggle and limitation removes the very friction that historically gave human achievement and pu

In [67]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [68]:
print(judge)

You are judging a competition between 10 competitors.
Each model has been given this question:

If a society achieves absolute abundance and eliminates all scarcity, what primary driver replaces economic necessity in giving human life a sense of meaning, and what paradox might arise from this transition? Answer in one sentence.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1
The intrinsic pursuit of creative self-actualization and personal mastery replaces economic necessity as the primary driver of meaning, generating the paradox that humanity must artificially manufacture struggle and constraint to find purpose in a world that has eradicated them.

# Response from competit

In [70]:
judge_messages = [{"role": "user", "content": judge}]

## And now for Grok!

Branded as "The most truth-seeking large language model in the world".. so let's use it as our LLM as a judge

In [73]:
# Judgement time!
# Grok is "The most truth-seeking large language model in the world."

model_name = "qwen2.5:7b"

response = ollama.chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)


{"results": ["1", "2", "3", "4", "6", "5", "7", "8", "10"]}


In [74]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gemini-3.8-flash
Rank 2: gemini-3.5-flash-lite
Rank 3: gemini-3.7-flash
Rank 4: gemini-3.6-flash
Rank 5: llama3.2:latest
Rank 6: llama3.2:latest
Rank 7: gemma3:270m
Rank 8: gemma3:270m
Rank 9: qwen2.5:7b


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>